# Template Pelatihan Model AI

Notebook ini untuk melatih model pendeteksi objek YOLO11s (dapat menggunakan model yang lain) agar dapat diintegrasikan secara optimal dengan aplikasi backend Veil.

## 1. Persiapan Environment
Jalankan cell di bawah ini untuk menginstal seluruh dependensi yang dibutuhkan (Ultralytics, Roboflow, ONNX). Pastikan Anda menggunakan environment yang memiliki akselerasi GPU (seperti NVIDIA T4 pada Cloud Platform).

In [ ]:
!pip install -q ultralytics roboflow onnx onnxslim onnxruntime albumentations

import os
import torch

print(f"Ketersediaan CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Aktif: {torch.cuda.get_device_name(0)}")
else:
    print("PERINGATAN: GPU tidak terdeteksi. Proses pelatihan akan memakan waktu yang lama.")

## 2. Unduh Dataset (Roboflow)
Masukkan code snippet dari proyek Roboflow Anda dengan format YOLOv8.

In [ ]:
from roboflow import Roboflow

# Ganti dengan kredensial Roboflow Anda
rf = Roboflow(api_key="MASUKKAN_API_KEY_ANDA_DI_SINI")
project = rf.workspace("NAMA_WORKSPACE_ANDA").project("NAMA_PROYEK_ANDA")
version = project.version(1)
dataset = version.download("yolov8")

## 3. Proses Pelatihan (Augmentasi Ketat)
Lakukan pelatihan dengan arsitektur bawaan YOLO11s. Template ini telah dikonfigurasi dengan metrik augmentasi ekstrim (seperti penghapusan sebagian piksel dan distorsi perspektif) untuk menyimulasikan ketidaksempurnaan visibilitas pada dunia nyata.

In [ ]:
from ultralytics import YOLO

# Memuat arsitektur dasar
model = YOLO('yolo11s.pt')

# Ubah parameter 'data' dengan direktori file .yaml dataset Anda
dataset_yaml = "NAMA_FOLDER_DATASET/data.yaml"

# Memulai proses iterasi pelatihan
model.train(
    data=dataset_yaml,       
    epochs=50,                # Sesuaikan batas iterasi (epochs) dengan skala dataset Anda
    imgsz=640,
    batch=8,
    device=0,
    
    # Konfigurasi augmentasi yang Veil gunakan
    mosaic=1.0,
    mixup=0.1,
    perspective=0.001,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    bgr=0.2,
    erasing=0.2,
)

## 4. Ekspor ke Format Produksi (ONNX)
Sistem *backend* Veil tidak direkomendasikan untuk membaca format PyTorch standar. Jalankan sel di bawah ini untuk mengekspor bobot model (*weights*) terbaik Anda ke format biner mesin lintas platform (.onnx).

In [ ]:
import glob

# Mencari hasil bobot terbaik (best.pt) secara otomatis pada log pelatihan terakhir
model_path = glob.glob("runs/detect/train*/weights/best.pt")

if model_path:
    # Mengambil rute pelatihan yang paling baru (train yang paling terakhir)
    latest_model_path = sorted(model_path)[-1]
    best_model = YOLO(latest_model_path)
    best_model.export(format='onnx', imgsz=640)
    print(f"\nEkspor selesai. Silakan unduh file .onnx dari folder: {latest_model_path.replace('.pt', '.onnx')}")
else:
    print("Bobot model (best.pt) tidak ditemukan.")